# Geometry from GVEC

TORAX solves the transport equations on the radius $\rho = \sqrt{\Phi/\Phi_b}$ alone; the shape of the flux surfaces enters through a few averages per surface. slowcooker computes them from a GVEC equilibrium, tokamak or stellarator, and hands them to TORAX through its CHEASE geometry format, in memory, without changing TORAX.

This page computes the averages of the W7-X equilibrium that comes with slowcooker and checks the chain on a circular tokamak against TORAX's own circular geometry.

In [ ]:
import logging

logging.disable(logging.INFO)  # TORAX and GVEC log every restart and state file at INFO

In [ ]:
import numpy as np
import plasma_plots

from slowcooker import circular_geometry, geometry_config, gvec_geometry
from slowcooker.data import gvec_equilibrium

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX

## W7-X

`gvec_equilibrium("w7x")` loads the W7-X standard configuration of GVEC's examples, solved at reduced resolution (five field periods, nearly no toroidal current). `gvec_geometry` evaluates GVEC's metric on a $(\theta, \zeta)$ grid of each surface and returns the averages as an xarray dataset: $V' = dV/d\rho$, $\langle|\nabla\rho|^2\rangle$, $\langle 1/R^2\rangle$, $\langle B^2\rangle$, the effective trapped fraction, $\iota$ (negative in GVEC's orientation; TORAX gets $|\iota|$) and more.

In [ ]:
w7x = gvec_equilibrium("w7x")
geometry = gvec_geometry(w7x, rho=np.linspace(0.02, 1.0, 40))
geometry

In [ ]:
titles = {
    "volume_derivative": r"$V' = dV/d\rho$ [m$^3$]",
    "grad_rho2": r"$\langle|\nabla\rho|^2\rangle$ [m$^{-2}$]",
    "rotational_transform": r"$\iota$",
    "trapped_fraction": "effective trapped fraction",
}
with plasma_plots.figure(2, 2, figsize=(9, 6), sharex=True) as fig:
    for index, (name, title) in enumerate(titles.items()):
        geometry[name].plasma.plot.lineout(x="rho", ax=fig[index], title=title)

## The TORAX geometry section

`geometry_config` turns the averages into the `geometry` section of a TORAX config. The poloidal flux comes from $\iota$, $\psi = \int \iota\, d\Phi$, so that TORAX's $q$ is $1/\iota$; the column of $F = RB_\varphi$ holds the value that makes TORAX's tokamak identity for $V'$ give GVEC's $V'$. The trapped fraction is taken from the averages, not from TORAX's tokamak formula.

In [ ]:
section = geometry_config(geometry, n_rho=25)
{key: value for key, value in section.items() if key != "chease_data"}

## A check on a circular tokamak

For concentric circular surfaces the averages are known in closed form. Built into TORAX's geometry, $V'$ agrees with TORAX's own circular model to round-off; $\langle 1/R^2\rangle$ differs because TORAX's circular model takes the large-aspect-ratio value $1/R_0^2$, while ours is exact.

In [ ]:
from torax._src.geometry import chease
from torax._src.geometry import circular_geometry as torax_circular

circle = circular_geometry(
    3.0, 1.0, 2.5, rotational_transform=0.5, rho=np.linspace(0.02, 1, 60)
)
config = geometry_config(circle, n_rho=25)
config.pop("model_name")
ours = chease.CheaseConfig(**config)._build_geometry()
theirs = torax_circular.CircularConfig(
    R_major=3.0, a_minor=1.0, B_0=2.5, n_rho=25, elongation_LCFS=1.0
)._build_geometry()
print(
    "largest relative difference of V':",
    float(np.max(np.abs(ours.vpr / theirs.vpr - 1))),
)
print(
    "<1/R^2> at the edge, ours and TORAX's:", float(ours.g3[-1]), float(theirs.g3[-1])
)